# Customer Churn Analysis

**Problem:** Find the number of churned customers per month. A customer is considered **churned** in a given month if they made a purchase in that month but do **not** make any purchase in the following month (i.e., no next-month record exists).

**Approach:**
1. Truncate each `order_date` to the month level (`month_ordered`)
2. Use `LEAD` partitioned by `cust_id` to find the next month the customer placed an order
3. If `nxt_month IS NULL`, the customer has no future purchase → they churned that month
4. Group by `month_ordered` and count churned customers

**Solutions below:** SQL (Cell 3) and PySpark (Cell 5)

In [0]:
%sql
-- ============================================================
-- Setup: Create & Populate Transactions1 Table
-- Columns: order_id, cust_id, order_date, amount
-- Sample data: 5 customers with purchases in Jan/Feb 2020
-- ============================================================

create table b_sql.b_practice.transactions1(
order_id int,
cust_id int,
order_date date,
amount int
);
insert into b_sql.b_practice.transactions1 values 
(1,1,'2020-01-15',150)
,(2,1,'2020-02-10',150)
,(3,2,'2020-01-16',150)
,(4,2,'2020-02-25',150)
,(5,3,'2020-01-10',150)
,(6,3,'2020-02-20',150)
,(7,4,'2020-01-20',150)
,(8,5,'2020-02-20',150)
;

In [0]:
%sql
-- Preview: Transactions1 Data
-- Verify inserted rows and inspect order dates and amounts
select * from b_sql.b_practice.transactions1

In [0]:
%sql
-- ============================================================
-- SQL Solution: Churned Customers Per Month
-- 1. CTE     → truncate order_date to month (month_ordered)
-- 2. CTE_2   → LAG/LEAD partitioned by cust_id to find
--             prev_month and nxt_month for each customer
-- 3. Final   → if nxt_month IS NULL → customer churned
--             (no future purchase); group by month and sum
-- ============================================================
WITH CTE AS (
    SELECT
        *,
        cast(DATE_TRUNC("month",  order_date) as date) AS month_ordered
    FROM b_sql.b_practice.transactions1
),

CTE_2 AS (
    SELECT
        cust_id,
        month_ordered,
        LAG(month_ordered) OVER (
            PARTITION BY cust_id
            ORDER BY month_ordered
        ) AS prev_month,
        LEAD(month_ordered) OVER (
            PARTITION BY cust_id
            ORDER BY month_ordered
        ) AS nxt_month
    FROM CTE
    GROUP BY cust_id, month_ordered
)

SELECT
    month_ordered,
    SUM(
        CASE
            WHEN nxt_month IS NULL THEN 1
            ELSE 0
        END
    ) AS no_churned_customers
FROM CTE_2
GROUP BY month_ordered
ORDER BY month_ordered;


In [0]:
# PySpark: Load Transactions1 into DataFrame
# Read transactions1 table from Unity Catalog and preview
df_transaction=spark.read.table("b_sql.b_practice.transactions1")
df_transaction.display()

In [0]:
# ============================================================
# PySpark Solution: Churned Customers Per Month
# 1. cte       → truncate order_date to month (month_ordered)
# 2. window    → partition by cust_id, order by month_ordered
# 3. cte_2     → distinct cust_id/month, add lag & lead columns
# 4. result    → if nxt_month IS NULL → churned (1), else 0
#                group by month_ordered, sum churned flags
# ============================================================
from pyspark.sql.functions import (
    col, date_trunc, lag, lead, sum, when
)
from pyspark.sql.window import Window

# CTE
cte = (
    df_transaction
    .withColumn(
        "month_ordered",
        date_trunc("month", col("order_date")).cast("date")
    )
)

# Window
window = Window.partitionBy("cust_id").orderBy("month_ordered")

# CTE_2
cte_2 = (
    cte
    .select(
        "cust_id",
        "month_ordered"
    )
    .distinct()
    .withColumn(
        "prev_month",
        lag("month_ordered").over(window)
    )
    .withColumn(
        "nxt_month",
        lead("month_ordered").over(window)
    )
)

# Final query
result = (
    cte_2
    .groupBy("month_ordered")
    .agg(
        sum(
            when(col("nxt_month").isNull(), 1)
            .otherwise(0)
        ).alias("no_churned_customers")
    )
    .orderBy("month_ordered")
)

result.display()
